# HGRIA - Hand Gesture Recognition for Interactive Applications
## Launch Notebook

```
┌─────────────────────────────────────────────────────────────┐
│                    ARCHITECTURE                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Browser (Frontend)     ngrok Tunnel      Colab Backend   │
│   ┌──────────────┐      ┌──────────┐     ┌──────────────┐   │
│   │  GitHub Pages │ ←─── │  HTTPS   │ ←── │  Flask +     │   │
│   │  / Vercel    │      │  Tunnel  │     │  MediaPipe   │   │
│   └──────────────┘      └──────────┘     └──────────────┘   │
│         │                                          │        │
│         │           Google Drive                    │        │
│         └──────────────┬───────────────────────────┘        │
│                        │ logs/                             │
└─────────────────────────────────────────────────────────────┘
```

### Prerequisites
- Google Account with Google Drive access
- ngrok account (free tier works)
- WebRTC-compatible browser (Chrome, Edge, Firefox)

### How it works
1. Backend runs Flask server on Colab with MediaPipe
2. ngrok creates HTTPS tunnel to expose backend
3. Frontend connects via WebSocket and sends webcam frames
4. Backend processes frames and sends gesture commands back

In [1]:
!pip install --no-cache-dir \
    "numpy==1.26.4" \
    "tensorflow==2.18.0" \
    "protobuf==4.25.3" \
    "mediapipe==0.10.21" \
    "opencv-contrib-python==4.11.0.86"

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Step 1: Mount Google Drive and verify project files
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

SOURCE_PATH = '/content/drive/MyDrive/HGRIA'
REQUIREMENTS_FILE = os.path.join(SOURCE_PATH, 'requirements.txt')

if not os.path.exists(REQUIREMENTS_FILE):
    raise FileNotFoundError(
        f"requirements.txt not found at {REQUIREMENTS_FILE}.\n"
        "Please ensure HGRIA project is saved in your Google Drive at:\n"
        "  /content/drive/MyDrive/HGRIA/"
    )

print(f"✓ Project files found at {SOURCE_PATH}")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# !pip uninstall -y mediapipe tensorflow tensorflow-cpu protobuf numpy jax jaxlib

In [ ]:
import numpy as np
import google.protobuf
import mediapipe as mp
import tensorflow as tf

print("NumPy:", np.__version__)
print("Protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)
print("MediaPipe OK")
print("TensorFlow OK")

NumPy: 1.26.4
Protobuf: 4.25.3
MediaPipe: 0.10.21
TensorFlow: 2.18.0
MediaPipe OK
TensorFlow OK


In [ ]:
# Step 2: Copy project to Colab and install dependencies
import shutil
import subprocess
import sys

DEST = '/content/HGRIA'

# Copy only if destination doesn't exist or user wants to refresh
if os.path.exists(DEST):
    print(f"Project already exists at {DEST}")
else:
    shutil.copytree(SOURCE_PATH, DEST)
    print(f"✓ Copied project to {DEST}")

# Add to Python path
sys.path.insert(0, DEST)
os.chdir(DEST)

# Install dependencies
result = subprocess.run(
    ['pip', 'install', '-q', '-r', 'requirements.txt'],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"Failed to install dependencies:\n{result.stderr}"
    )

print("✓ Dependencies installed successfully")

NameError: name 'SOURCE_PATH' is not defined

In [ ]:
# Step 3: Configure ngrok authentication
import getpass

# Install pyngrok if not present
subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
from pyngrok import ngrok

# Get ngrok auth token (optional but recommended)
print("Enter your ngrok authtoken (from https://dashboard.ngrok.com/auth)")
print("Press Enter to skip (anonymous tunnel may disconnect)")

authtoken = getpass.getpass(prompt='Authtoken: ')

if authtoken:
    ngrok.set_auth_token(authtoken)
    print("✓ ngrok authenticated")
else:
    print("⚠ Anonymous tunnel - connection may be unstable")

Enter your ngrok authtoken (from https://dashboard.ngrok.com/auth)
Press Enter to skip (anonymous tunnel may disconnect)
Authtoken: ··········
✓ ngrok authenticated


In [ ]:
# Step 4: Start ngrok tunnel and display connection info
import json

# Start ngrok tunnel
tunnel = ngrok.connect(5000, "http")

# Convert HTTP URL to HTTPS
ngrok_url = tunnel.public_url.replace('http://', 'https://')

print("=" * 60)
print("🔗 NGROK TUNNEL READY")
print("=" * 60)
print(f"\nBackend URL: {ngrok_url}")

# Generate frontend integration snippet
frontend_script = f'<script>window.HGRIA_BACKEND_URL="{ngrok_url}";<\/script>'
print(f"\nPaste this in your frontend HTML (before Socket.IO loads):")
print(f"\n{frontend_script}")

# Generate frontend URL (assuming GitHub Pages or Vercel)
frontend_base = "https://qtannguyen-researcher.github.io/HGRIA"
frontend_url = f"{frontend_base}?server={ngrok_url}"

print(f"\nOr open Frontend directly with:")
print(f"\n{frontend_url}")
print("\n" + "=" * 60)

<>:16: SyntaxWarning: invalid escape sequence '\/'
<>:16: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipykernel_1455/908728183.py:16: SyntaxWarning: invalid escape sequence '\/'
  frontend_script = f'<script>window.HGRIA_BACKEND_URL="{ngrok_url}";<\/script>'


🔗 NGROK TUNNEL READY

Backend URL: https://stylized-stark-proofing.ngrok-free.dev

Paste this in your frontend HTML (before Socket.IO loads):

<script>window.HGRIA_BACKEND_URL="https://stylized-stark-proofing.ngrok-free.dev";<\/script>

Or open Frontend directly with:

https://qtannguyen-researcher.github.io/HGRIA?server=https://stylized-stark-proofing.ngrok-free.dev



In [ ]:
# Step 5: Configure and patch config (Colab + local compatible)
import json
import sys
from pathlib import Path

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

# Resolve project root robustly:
#   - Colab: DEST was set to '/content/HGRIA' in Step 2
#   - Local: notebook is at <project_root>/notebooks/, so root is one level up
if IN_COLAB:
    PROJECT_ROOT = Path(DEST)
else:
    # Path.cwd() when running via Jupyter points to the notebook directory
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

# Ensure project root is on sys.path so 'backend' package is importable
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

config_path = str(PROJECT_ROOT / 'config' / 'config.json')

# Load existing config
with open(config_path, 'r') as f:
    config = json.load(f)

# Apply environment-specific patches
config['camera']['colab_mode'] = IN_COLAB
config['server']['cors_origins'] = '*'
config['logging']['log_to_file'] = True

if IN_COLAB:
    config['logging']['log_file_path'] = '/content/drive/MyDrive/HGRIA/logs/'
else:
    log_dir = PROJECT_ROOT / 'logs'
    log_dir.mkdir(exist_ok=True)
    config['logging']['log_file_path'] = str(log_dir) + '/'

# Write patched config back
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Project root : {PROJECT_ROOT}")
print(f"✓ Environment  : {'Colab' if IN_COLAB else 'Local'}")
print("✓ Config patched:")
print(f"  - colab_mode   : {config['camera']['colab_mode']}")
print(f"  - cors_origins : {config['server']['cors_origins']}")
print(f"  - log_file_path: {config['logging']['log_file_path']}")

✓ Project root : /home/qtannguyen/projects/researcher/HGRIA
✓ Environment  : Local
✓ Config patched:
  - colab_mode   : False
  - cors_origins : *
  - log_file_path: /home/qtannguyen/projects/researcher/HGRIA/logs/


In [ ]:
# Step 6: Start the HGRIA Server (blocking)
# This cell will keep running until interrupted
from backend.main import SystemOrchestrator

print("Starting HGRIA Backend Server...")
print(f"Server running at: {ngrok_url}")
print("\nPress Stop button to terminate the server")
print("-" * 40)

# Start the orchestrator (blocking)
orchestrator = SystemOrchestrator(config_path)
orchestrator.start()

Starting HGRIA Backend Server...
Server running at: https://stylized-stark-proofing.ngrok-free.dev

Press Stop button to terminate the server
----------------------------------------
{"timestamp": "2026-08-16T12:31:40.901+00:00", "level": "INFO", "module": "orchestrator", "event": "startup_begin"}
{"timestamp": "2026-08-16T12:31:40.902+00:00", "level": "INFO", "module": "orchestrator", "event": "config_loaded"}


{"timestamp": "2026-08-16T12:31:40.903+00:00", "level": "INFO", "module": "orchestrator", "event": "mediapipe_init_start"}
{"timestamp": "2026-08-16T12:31:41.023+00:00", "level": "INFO", "module": "hand_detector", "event": "mediapipe_warmup_complete", "warmup_ms": 45.17}
{"timestamp": "2026-08-16T12:31:41.023+00:00", "level": "INFO", "module": "orchestrator", "event": "mediapipe_ready"}


I0000 00:00:1786883500.959038   49815 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1786883500.975599   53967 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.4), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786883500.995074   53932 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786883501.001913   53931 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


{"timestamp": "2026-08-16T12:31:41.233+00:00", "level": "INFO", "module": "orchestrator", "event": "camera_ready"}
{"timestamp": "2026-08-16T12:31:41.392+00:00", "level": "INFO", "module": "orchestrator", "event": "flask_server_ready"}
{"timestamp": "2026-08-16T12:31:41.402+00:00", "level": "WARNING", "module": "pipeline_runner", "event": "dynamic_gestures_init_failed", "error": "No module named 'onnxruntime'"}
{"timestamp": "2026-08-16T12:31:41.403+00:00", "level": "INFO", "module": "pipeline_runner", "event": "pipeline_started"}
{"timestamp": "2026-08-16T12:31:41.403+00:00", "level": "INFO", "module": "orchestrator", "event": "pipeline_started"}
Server URL: http://0.0.0.0:5000{"timestamp": "2026-08-16T12:31:41.801+00:00", "level": "INFO", "module": "orchestrator", "event": "ngrok_tunnel_opened", "url": "https://stylized-stark-proofing.ngrok-free.dev"}

Public URL: https://stylized-stark-proofing.ngrok-free.dev
 * Serving Flask app 'backend.app'
 * Debug mode: off
{"timestamp": "2026-

/home/qtannguyen/.local/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


...

Error on request:
Traceback (most recent call last):
  File "/home/qtannguyen/.local/lib/python3.10/site-packages/werkzeug/serving.py", line 371, in run_wsgi
    execute(self.server.app)
  File "/home/qtannguyen/.local/lib/python3.10/site-packages/werkzeug/serving.py", line 337, in execute
    write(b"")
  File "/home/qtannguyen/.local/lib/python3.10/site-packages/werkzeug/serving.py", line 262, in write
    assert status_set is not None, "write() before start_response"
AssertionError: write() before start_response


{"timestamp": "2026-08-16T12:35:34.806+00:00", "level": "WARNING", "module": "state_manager", "event": "invalid_transition", "current": "Disconnected", "transition_event": "no_hand_detected"}
{"timestamp": "2026-08-16T12:35:34.869+00:00", "level": "WARNING", "module": "state_manager", "event": "invalid_transition", "current": "Disconnected", "transition_event": "no_hand_detected"}
{"timestamp": "2026-08-16T12:35:34.927+00:00", "level": "WARNING", "module": "state_manager", "event": "invalid_transition", "current": "Disconnected", "transition_event": "no_hand_detected"}
{"timestamp": "2026-08-16T12:35:34.988+00:00", "level": "WARNING", "module": "state_manager", "event": "invalid_transition", "current": "Disconnected", "transition_event": "no_hand_detected"}
{"timestamp": "2026-08-16T12:35:35.047+00:00", "level": "WARNING", "module": "state_manager", "event": "invalid_transition", "current": "Disconnected", "transition_event": "no_hand_detected"}
{"timestamp": "2026-08-16T12:35:35.108+0

t=2026-08-17T08:39:13+0700 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=9d200b41f0e5 clientid=5bb6f074aa03c594bcc9af851e339c36
t=2026-08-17T08:39:13+0700 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=dce3e82a9c24 err="session closed"


{"timestamp": "2026-08-17T01:39:13.492+00:00", "level": "INFO", "module": "state_manager", "event": "state_transition", "old": "Searching", "new": "Disconnected", "transition_event": "client_disconnected"}


Error on request:
Traceback (most recent call last):
  File "/home/qtannguyen/.local/lib/python3.10/site-packages/werkzeug/serving.py", line 371, in run_wsgi
    execute(self.server.app)
  File "/home/qtannguyen/.local/lib/python3.10/site-packages/werkzeug/serving.py", line 337, in execute
    write(b"")
  File "/home/qtannguyen/.local/lib/python3.10/site-packages/werkzeug/serving.py", line 262, in write
    assert status_set is not None, "write() before start_response"
AssertionError: write() before start_response


## Post-Launch Instructions

### Accessing the Frontend

After the server starts, open your browser and navigate to:

```
https://your-username.github.io/HGRIA/?server=<NGROK_URL>
```

Or paste the `window.HGRIA_BACKEND_URL` script into the frontend HTML before Socket.IO.

### If ngrok URL Changes

1. Stop the server (interrupt the cell above)
2. Re-run cells 4, 5, and 6 in sequence
3. Update the frontend with the new URL

### Troubleshooting

| Issue | Solution |
|-------|----------|
| Colab session timeout | Re-run cell 6 (server restart is automatic) |
| ngrok URL changed | Re-run cells 4, 5, 6 and update frontend |
| Webcam denied | Use keyboard fallback (Arrow keys, Space, P, S) |
| High latency | Check Colab GPU availability |

### Keyboard Controls (Fallback)
- Arrow Keys: Move
- Space: Jump
- P: Pause
- S: Speed Boost
- Enter: Confirm